# D10 — 1단계 NER 500만 학습 (경제 사전학습판 → 7태그), Colab

**무엇**: 경제 사전학습한 모델에 1단계 학습 세트 500만 문장(7태그, span 규칙 v3b)을 1에폭 학습한다. 120만(dev 0.7875)과 같은 dev·test로 채점.
서버에서는 roberta-base → roberta-large 를 차례로 돌리므로 **여기서는 mmbert 만 실행**한다(large 는 서버가 실패할 때를 위한 예비).

| MODEL | 출발 가중치 | 입력 | 학습률 | 예상(추정) |
|---|---|---|---|---|
| `large` | 경제 사전학습 roberta-large | 원문 | 2e-5 (1단계 large 실험과 같음) | A100 약 5~9시간 |
| `mmbert` | 경제 사전학습 mmBERT (1에포크) | 형태소 | 5e-5 | A100 약 4~7시간 |

**런타임**: A100 GPU(40GB 충분) · RAM 표준. 비슷한 길이끼리 묶어 학습해 패딩 낭비를 줄였다(그래서 1단계보다 빠를 것으로 봄, 4번 셀이 남은 시간을 출력).
**준비물 (내 드라이브 `ner_stage1_5m/`)**: `src/`(ner_lib·evaluate·train_big), `data/`(dev·test), `enc/roberta_5000k`·`enc/mmbert_morph_5000k`(미리 토큰화, 검사 통과), `init/dapt_roberta_large_econ5m`. mmBERT 출발 가중치는 `ner_dapt/models/` 것을 쓴다.
**데이터 검사**: 학습 세트 500만은 독립 검증기 통과(줄 수·중복·기존 120만 앞부분 일치·평가/골드셋/Claude 표본과 겹침 0·span 형식·원본 라벨·v3b 규칙·분포·원본 대조). dev·test도 규칙·중복·겹침 검사 통과. 토큰화는 앞 2,000문장 재계산과 일치.
**순서**: 위에서부터 모두 실행. 끊기면 1~4번 다시 → 드라이브 체크포인트(2만 스텝마다)부터 이어서. 6번은 저장 확인·채점이 끝났을 때만 런타임 반납.
**드라이브 공간**: 체크포인트가 large 약 8GB·mmBERT 약 7GB 필요. 휴지통의 옛 체크포인트(약 58GB)를 비워 두면 안전.


In [ ]:
# 1. 고르기 + GPU 확인
MODEL = 'mmbert'     # 'mmbert' (roberta-large 는 서버에서 돌림: 10/25 결정)
import torch, subprocess
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다'
p = torch.cuda.get_device_properties(0); print(p.name, f'{p.total_memory/2**30:.0f}GB', 'bf16', torch.cuda.is_bf16_supported())
assert torch.cuda.is_bf16_supported(), 'T4 등 bf16 미지원 GPU — A100으로 바꾸세요'

In [ ]:
# 2. 드라이브 연결·파일 확인 → 빠른 로컬 디스크로 복사 (드라이브 위에서는 배열을 바로 못 읽음)
from google.colab import drive
drive.mount('/content/drive')
import os, json, shutil, time
ROOT = '/content/drive/MyDrive/ner_stage1_5m'
CFG = {'large':  dict(enc='roberta_5000k', init=f'{ROOT}/init/dapt_roberta_large_econ5m', lr=2e-5, pretok='none', exp='S5_dapt-roberta-large_5000k'),
       'mmbert': dict(enc='mmbert_morph_5000k', init='/content/drive/MyDrive/ner_dapt/models/dapt_mmbert_econ5m_morph', lr=5e-5, pretok='morph', exp='S5_dapt-mmbert-morph_5000k')}[MODEL]
L = '/content/local'; os.makedirs(L, exist_ok=True)
for name, src in [('enc', f"{ROOT}/enc/{CFG['enc']}"), ('init', CFG['init']), ('src', f'{ROOT}/src'), ('data', f'{ROOT}/data')]:
    t = time.time(); dst = f'{L}/{name}'
    if not os.path.exists(dst): shutil.copytree(src, dst)
    print(name, sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(dst) for f in fs) // 2**20, 'MB', f'{time.time()-t:.0f}초')
meta = json.load(open(f'{L}/enc/meta.json')); print(meta)
assert meta['check_first_2000'] == '2000/2000' and meta['sentences'] == 5_000_000
assert os.path.getsize(f'{L}/enc/ids.bin') == meta['tokens'] * (2 if meta['dtype'] == 'uint16' else 4), '토큰 파일 크기가 다름(동기화 덜 됨)'

In [ ]:
# 3. 패키지
!pip install -q kiwipiepy pytorch-crf
import transformers; print('transformers', transformers.__version__)

In [ ]:
# 4. 학습 (끊기면 1~4번 다시 실행 → 이어서). 체크포인트·결과는 드라이브 ROOT/out 에 저장
!cd {L} && python src/train_big.py --enc {L}/enc --init {L}/init --exp {CFG['exp']} --pretok {CFG['pretok']} --lr {CFG['lr']}     --epochs 1 --batch 16 --save_steps 20000 --out_root {ROOT}/out --dev {L}/data/dev.jsonl --test {L}/data/test.jsonl --workers 4

In [ ]:
# 5. 결과 확인
r = json.load(open(f"{ROOT}/out/reports/results/{CFG['exp']}.json", encoding='utf-8'))
print('저장 확인 텐서', r['weights_verified'], '· 학습', round(r['train_sec']/3600, 2), '시간 ·', r['gpu'])
print('dev F1', r['dev']['ALL']['f1'], '· test F1', r['test']['ALL']['f1'], '· test macro', r['test']['MACRO_F1'])
print('참고: 120만(roberta-base, 사전학습 없음) dev 0.7875 · test 0.7773')
OK = bool(r.get('done')) and os.path.exists(f"{ROOT}/out/ner_models/{CFG['exp']}/model.safetensors")
print('OK =', OK)

In [ ]:
# 6. 런타임 반납 (5번이 OK일 때만)
from google.colab import runtime
if OK: print('반납'); runtime.unassign()
else: print('결과 확인 실패 → 반납하지 않음')